# RL-Based Tsunami Alert Decision System\n\nThis notebook demonstrates training, evaluation, log inspection, and sample greedy trajectories for the custom tsunami warning RL environment.

In [ ]:
from pathlib import Path\nimport sys\nimport json\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nROOT = Path.cwd()\nif not (ROOT / 'src').exists():\n    ROOT = ROOT.parent\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nfrom config import ProjectConfig\nfrom src.agent import QLearningAgent\nfrom src.environment import TsunamiAlertEnvironment\nfrom src.trainer import Trainer\nfrom src.evaluator import Evaluator

In [ ]:
config = ProjectConfig(training_episodes=800, evaluation_episodes=120, random_seed=42)\nenv = TsunamiAlertEnvironment(config=config, seed=config.random_seed)\nagent = QLearningAgent(\n    state_size=config.state_size,\n    action_size=config.action_size,\n    alpha=config.alpha,\n    gamma=config.gamma,\n    epsilon=config.epsilon,\n    epsilon_decay=config.epsilon_decay,\n    min_epsilon=config.min_epsilon,\n    seed=config.random_seed,\n)\ntrainer = Trainer(config=config, environment=env, agent=agent)\ntrain_summary = trainer.train()\ntrain_summary

In [ ]:
training_log = pd.read_csv(config.logs_dir / 'training_history.csv')\ntraining_log.head()

In [ ]:
plt.figure(figsize=(10, 4))\nplt.plot(training_log['episode'], training_log['total_reward'], linewidth=1.0)\nplt.title('Training Reward vs Episode')\nplt.xlabel('Episode')\nplt.ylabel('Reward')\nplt.grid(alpha=0.3)\nplt.show()

In [ ]:
eval_env = TsunamiAlertEnvironment(config=config, seed=config.random_seed)\neval_agent = QLearningAgent(\n    state_size=config.state_size,\n    action_size=config.action_size,\n    alpha=config.alpha,\n    gamma=config.gamma,\n    epsilon=0.0,\n    epsilon_decay=1.0,\n    min_epsilon=0.0,\n    seed=config.random_seed,\n)\neval_agent.load_q_table(config.models_dir / 'q_table.npy')\nevaluator = Evaluator(config=config, environment=eval_env, agent=eval_agent)\neval_summary = evaluator.evaluate(episodes=120)\neval_summary

In [ ]:
with (config.logs_dir / 'evaluation_summary.json').open('r', encoding='utf-8') as f:\n    summary_json = json.load(f)\nsummary_json

In [ ]:
sample_trajectories = []\nfor episode in range(3):\n    state_idx = eval_env.reset()\n    done = False\n    trace = []\n    while not done:\n        action = eval_agent.choose_action(state_idx, training=False)\n        next_state_idx, reward, done, info = eval_env.step(action)\n        trace.append({\n            'state_idx': state_idx,\n            'action': action,\n            'action_name': info['action_meaning'],\n            'reward': reward,\n            'next_state_idx': next_state_idx,\n            'actual_risk': info['actual_risk_level'],\n            'done': done,\n        })\n        state_idx = next_state_idx\n    sample_trajectories.append(trace)\n\nsample_trajectories